In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss
from sklearn.calibration import calibration_curve

import xgboost as xgb
import optuna

In [ ]:
#Loss Functions

def multiclass_brier(y_true, probs):
    y_onehot = np.eye(3)[y_true]
    return np.mean(np.sum((probs-y_onehot)**2, axis=1))

def rps_score(y_true, probs):

    actual = np.eye(3)[y_true]

    pred_cum = np.cumsum(probs[:, :-1], axis=1)
    actual_cum = np.cumsum(actual[:, :-1], axis=1)

    return np.mean(
        np.sum((pred_cum - actual_cum)**2, axis=1)
    )

def calibration_score_plot(y_test, probs, n_bins=10):

    calibration_results = {}

    plt.figure(figsize=(8, 6))

    for class_id, name in enumerate(
        ["Home win", "Draw", "Away win"]
    ):

        # Convert multiclass labels to binary for this class
        y_binary = (y_test == class_id).astype(int)

        prob_true, prob_pred = calibration_curve(
            y_binary,
            probs[:, class_id],
            n_bins=n_bins,
            strategy="uniform"
        )

        calibration_results[name] = {
            "predicted_probability": prob_pred,
            "observed_frequency": prob_true
        }

        plt.plot(
            prob_pred,
            prob_true,
            marker="o",
            label=name
        )

    # Perfect calibration line
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    plt.title("Calibration Curve: Football Win/Draw/Loss Model")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed frequency")
    plt.legend()
    plt.grid(True)
    plt.show()

    return calibration_results

In [ ]:
features = pd.read_csv("stored_features/match_features.csv")

In [ ]:
features.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score', 'winner',
       'winner_code', 'home_neutral', 'home_ppg_last_5', 'home_ppg_last_10',
       'home_avg_goals_scored_last_5', 'home_avg_goals_conceded_last_5',
       'home_clean_sheets_rate_last_5', 'home_avg_goal_difference_last_5',
       'home_avg_goal_difference_last_10', 'home_btts',
       'home_btts_rate_last_5', 'home_failed_score_rate_last_5',
       'home_over_2_5_rate_last_5', 'home_over_3_5_rate_last_5',
       'home_under_1_5_rate_last_5', 'home_days_since_last_game',
       'home_goal_difference_trend', 'home_momentum', 'home_pi_home_rating',
       'home_pi_away_rating', 'home_pi_expected_gd', 'home_pi_diff',
       'away_ppg_last_5', 'away_ppg_last_10', 'away_avg_goals_scored_last_5',
       'away_avg_goals_conceded_last_5', 'away_clean_sheets_rate_last_5',
       'away_avg_goal_difference_last_5', 'away_avg_goal_difference_last_10',
       'away_btts', 'away_btts_rate_last_5', 'away_failed_score_rate

In [32]:
features = features.loc[(features["date"] < "2026-06-01") & (features["date"] > "1920-01-01")].sort_values(by="date", ascending=True)
features = features.drop(columns=["date", "home_team", "away_team", "home_score", "away_score", "winner"])
#features = features.to_numpy()

In [ ]:
scaler = StandardScaler()
TSS = TimeSeriesSplit(n_splits=5)
split = TSS.split(features)
X_train, X_test, Y_train, Y_test = None, None, None, None
for i, (train_index, test_index) in enumerate(split):
    # print(f"Split {i+1}:")
    # print(f"Train indices: {train_index}")
    # print(f"Test indices: {test_index}")
    X_train = features.iloc[train_index]
    X_test = features.iloc[test_index]
Y_train = X_train["winner_code"]
Y_test = X_test["winner_code"]
X_train = X_train.drop(columns=["winner_code"])
X_test = X_test.drop(columns=["winner_code"])

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# X_train = X_train.to_numpy()
# X_test = X_test.to_numpy()
# Y_train = Y_train.to_numpy()
# Y_test = Y_test.to_numpy()

In [ ]:

def objective(trial):

    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "random_state": 42,

        "n_estimators": trial.suggest_int(
            "n_estimators",
            200,
            1000
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.15,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            8
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            20
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1,
            20,
            log=True
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5
        )
    }


    model = xgb.XGBClassifier(**params)

    model.fit(
        X_train,
        Y_train,
        eval_set=[
            (X_train, Y_train),
            (X_test, Y_test)
        ],
        verbose=False
    )


    probs = model.predict_proba(X_test)

    results ={
        "log_loss": log_loss(Y_test, probs),
    }

    return score

In [ ]:
study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=100
)

In [ ]:
print(study.best_params)

In [ ]:
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=5,
    reg_alpha=0.1,
    random_state=42
)

In [ ]:
print(Y_train)

[-1.51134537  1.08344331 -1.51134537 ...  1.08344331  1.08344331
  1.08344331]


In [51]:
trained_model = model.fit(X=X_train, y=Y_train, eval_set=[(X_train, Y_train), (X_test, Y_test)], verbose=True)

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got [-1.51134537 -0.21395103  1.08344331]